# PhoBERT v8: v7 recipe + 1,000 comments from more sources, best of 3 seeds

Setup: **GPU T4**, **Internet On**, secret `HF_TOKEN`.
**Input:** one private dataset with `v8_train.csv` and `v8_val.csv` (columns `text,label`, guideline v3.3):
the v7 rows plus 1,000 new comments (500 VnExpress news comments from ViCTSD, 300 VOZ forum posts, 200 more
YouTube comments), cleaned of @user tags, emoji and very short comments. 10% of the new rows went to validation.

Same recipe as v7 (continue v3 for 3 epochs, lr 1e-5), three seeds, the best validation F1-macro is kept and
pushed to the **`v8` branch** (`main` and `v7` are untouched). The held-out test half is not on Kaggle.
Takes about 20 minutes.

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/repo  # a re-run would otherwise keep the old checkout
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyvi emoji
import glob, os, shutil, json
os.makedirs("data/adapt_v6", exist_ok=True)
for split, name in [("train", "v8_train.csv"), ("val", "v8_val.csv")]:
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert len(matches) == 1, f"Expected exactly one {name}, found: {matches}"
    shutil.copy(matches[0], f"data/adapt_v6/{split}.csv")
shutil.copy("data/adapt_v6/val.csv", "data/adapt_v6/test.csv")  # the real test half never leaves the author's machine
os.makedirs("reports", exist_ok=True)


In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
from huggingface_hub import whoami
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
runs = {}
for seed in [42, 1, 2]:
    !python scripts/train_phobert.py --model {repo_id} --data-dir data/adapt_v6 --output-dir models/v8_s{seed} --epochs 3 --lr 1e-5 --batch-size 16 --max-len 128 --seed {seed} --tag _v8_s{seed} --no-push
    runs[seed] = json.load(open(f"reports/phobert_results_v8_s{seed}.json"))["val_f1_macro"]
best = max(runs, key=runs.get)
print("validation F1-macro per seed:", runs, "-> best seed:", best)


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_branch(repo_id, branch="v8", exist_ok=True)
api.upload_folder(folder_path=f"models/v8_s{best}/best", repo_id=repo_id, revision="v8", commit_message=f"v8: v7 data + 1,000 comments from news, forum and YouTube (seed {best}, best of 3 on validation)")
print(f"https://huggingface.co/{repo_id}/tree/v8")


Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps({"val_f1_macro_per_seed": runs, "best_seed": best, "best": json.load(open(f"reports/phobert_results_v8_s{best}.json"))}, ensure_ascii=False))
